# NYC Rideshare Profit Zone Recommender — Data Pipeline

This notebook covers the full data pipeline for the NYC Rideshare Profit Zone Recommender project — from raw data acquisition through to the final curated feature set used for modelling.

NYC's 2025 Congestion Pricing Program shifted FHV demand in ways that made traditional hotspot intuition unreliable. This project builds a zone-hour level profit score and trains a model to recommend the Top-5 most profitable zones for drivers each hour.

**Data sources used:**

- NYC TLC High-Volume FHV Trip Records (~120M trips/year)
- NOAA Central Park hourly weather data
- US public holiday calendar (`holidays` library)
- NYC TLC taxi zone shapefiles (263 zones)

**Train/validation/test split:**

- Train: Jan–Apr 2024
- Validation: May–Jun 2024
- Test: Jan–Jun 2025 (unseen year, post congestion pricing)

---


**Pipeline overview:**

| Section               | Output                                    |
| --------------------- | ----------------------------------------- |
| 1. Data Acquisition   | Raw parquet files + zone shapefiles       |
| 2. Trip Processing    | Zone–hour features + profit score         |
| 3. Weather Processing | Hourly weather features                   |
| 4. Calendar Features  | Hourly holiday + weekend flags            |
| 5. Feature Join       | Curated feature table (no target leakage) |
| 6. Zone Geospatial    | Zone centroids + GeoJSON for maps         |


## 1. Data Acquisition

All trip data is sourced directly from the [NYC TLC Trip Record Data portal](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). We focus exclusively on High-Volume FHV records (Uber, Lyft, Via) rather than yellow or green taxi data, as HVFHV trips dominate modern rideshare demand in NYC.

We download January–June for both 2024 and 2025. Using the same calendar months across both years controls for seasonal effects — ensuring that model generalisation to 2025 reflects the real policy shift rather than differences in summer vs winter demand.


### 1.1 Environment Setup


In [1]:
import os
import sys
import pathlib

PROJECT_ROOT = pathlib.Path(os.getcwd()).parent  # one level up from notebooks/
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
EXT_DIR = PROJECT_ROOT / "data" / "external"

sys.path.append(str(SCRIPTS_DIR))

### 1.2 Downloading Trip & Zone Data

The `tlc_io` helper script (in `/scripts`) handles idempotent downloads — it skips files that already exist locally and logs a manifest of what was fetched. This keeps the pipeline re-runnable without re-downloading ~6GB of data each time.


In [ ]:
from tlc_io import download_zones, download_trips

# Download taxi zone lookup CSV and shapefiles (used later for geospatial joins)
download_zones(unzip=True)

# Download HVFHV trip records for Jan–Jun 2024 and 2025
YEARS = ['2024', '2025']
MONTHS = list(range(1, 7))   # January through June
TYPES = ['fhv_hv']          # High-Volume FHV only

trip_results = download_trips(types=TYPES, years=YEARS, months=MONTHS)

# Summary: how many files downloaded and their sizes
print(f"Total files: {len(trip_results)}")
for r in trip_results:
    size_mb = r['size_bytes'] / 1e6
    print(f"  {r['year']}-{r['month']:>2}  {size_mb:>7.1f} MB  [{r['status']}]")

## 2. Trip Data Processing (HVFHV)

This section processes the raw HVFHV trip records into a clean **zone × hour** feature table ready for modelling.

The core challenge here is scale and quality. Each year contains ~120 million raw trip records, many of which contain implausible values — negative durations, impossibly fast speeds, or fare amounts that suggest recording errors rather than real trips. A multi-stage filtering pipeline removes these while preserving operationally meaningful edge cases like airport surcharges and storm-hour demand spikes.

The output of this section is a single aggregated table at the **zone–hour level**, with lag features and a custom profit score — the modelling target.


### 2.1 Spark Session

PySpark is used throughout to handle the volume of data. Key configurations:

- `spark.driver.memory = 8g` — required for aggregations over ~120M rows
- `spark.sql.session.timeZone = America/New_York` — ensures all timestamps are interpreted in NYC local time, not UTC, which matters for hour-of-day features
- Adaptive query execution enabled to handle partition skew during zone-level aggregations


In [ ]:
from scripts.spark_session import get_spark
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = get_spark(app_name="01_process_fhvhv")

### 2.2 Schema Normalisation

The 2024 and 2025 datasets have a minor schema difference: the `cbd_congestion_fee` column (introduced with the Congestion Pricing Program) is present in 2025 records but absent in 2024. To allow both years to be processed through the same pipeline, we backfill this column with `0.0` for 2024 records.

Y/N flag columns (e.g. `shared_request_flag`, `wav_request_flag`) are also cast to boolean for consistency.


In [ ]:
def lowercase_cols(df):
    return df.toDF(*[c.lower() for c in df.columns])


def ny_to_bool_col(col):
    """Convert TLC Y/N string flags to boolean."""
    return (F.when(F.col(col) == "Y", True)
             .when(F.col(col) == "N", False)
             .otherwise(None))


def normalize_fhvhv(df, year: int):
    """Lowercase columns, backfill cbd_congestion_fee for 2024, cast Y/N flags to boolean."""
    df = lowercase_cols(df)

    # 2024 data predates the Congestion Pricing Program — backfill with 0.0
    if year == 2024 and "cbd_congestion_fee" not in df.columns:
        df = df.withColumn("cbd_congestion_fee", F.lit(0.0).cast("double"))

    flags = ["shared_request_flag", "shared_match_flag",
             "access_a_ride_flag", "wav_request_flag", "wav_match_flag"]
    for f in flags:
        if f in df.columns:
            df = df.withColumn(f, ny_to_bool_col(f))

    return df

### 2.3 Feature Derivation

Three derived features are computed from the raw timestamps and distance fields:

- **`trip_minutes`** — trip duration in minutes. Derived from dropoff minus pickup timestamps where available; falls back to the raw `trip_time` field (seconds ÷ 60) otherwise.
- **`pickup_hour_local`** — the NYC-local hour bucket for each trip, used as the aggregation key. Using local time (not UTC) is critical — a trip at midnight NYC time is UTC-4 or UTC-5 depending on daylight saving, and misassigning it would corrupt the hour-of-day signal.
- **`avg_speed_mph`** — derived as `(trip_miles / trip_minutes) × 60`. Nulled out when duration is zero to avoid division errors.


In [ ]:
def add_time_columns_local(df):
    """Cast to NYC-local timestamps and derive pickup hour bucket."""
    if "pickup_datetime" in df.columns:
        df = df.withColumn("pickup_ts_local",
                           F.col("pickup_datetime").cast("timestamp"))
        df = df.withColumn("pickup_hour_local",
                           F.date_trunc("hour", F.col("pickup_ts_local")))
    if "dropoff_datetime" in df.columns:
        df = df.withColumn("dropoff_ts_local",
                           F.col("dropoff_datetime").cast("timestamp"))
    return df


def add_trip_minutes(df):
    """Derive trip duration in minutes from timestamps or fallback trip_time field."""
    if "pickup_ts_local" in df.columns and "dropoff_ts_local" in df.columns:
        df = df.withColumn(
            "trip_minutes",
            (F.col("dropoff_ts_local").cast("long") -
             F.col("pickup_ts_local").cast("long")) / 60.0
        )
    elif "trip_time" in df.columns:
        df = df.withColumn("trip_minutes", (F.col("trip_time") / 60.0))
    else:
        df = df.withColumn("trip_minutes", F.lit(None).cast("double"))
    return df


def add_trip_avg_speed(df, miles_col="trip_miles",
                       minutes_col="trip_minutes", out_col="avg_speed_mph"):
    """Compute average speed in mph. Returns null when duration is zero or missing."""
    if miles_col in df.columns and minutes_col in df.columns:
        df = df.withColumn(
            out_col,
            F.when(
                F.col(minutes_col).isNotNull() &
                (F.col(minutes_col) > 0) &
                F.col(miles_col).isNotNull(),
                F.col(miles_col) * F.lit(60.0) / F.col(minutes_col)
            ).otherwise(F.lit(None).cast("double"))
        )
    return df

### 2.4 Fare Column Handling

Fare-related columns occasionally contain nulls (e.g. missing airport fee when no airport was involved). These are filled with `0.0` rather than dropped, since a null airport fee is semantically zero — not a missing observation. Dropping these rows would incorrectly discard valid trips.


In [ ]:
def fill_amount_nulls_with_zero(df):
    """Cast fare/fee columns to double and fill nulls with 0.0."""
    amount_cols = [
        "base_passenger_fare", "tolls", "bcf", "sales_tax",
        "congestion_surcharge", "airport_fee", "tips",
        "driver_pay", "cbd_congestion_fee"
    ]
    for c in amount_cols:
        if c in df.columns:
            df = df.withColumn(c, F.coalesce(
                F.col(c).cast("double"), F.lit(0.0)))
    return df

### 2.5 Data Quality Filtering

Filtering happens in two stages:

**Stage 1 — Fatal filters (hard rules):** Records that are physically impossible or violate TLC data integrity rules are removed unconditionally. Thresholds are set conservatively to avoid discarding legitimate edge cases.

| Rule                | Threshold                           | Rationale                                                                          |
| ------------------- | ----------------------------------- | ---------------------------------------------------------------------------------- |
| Trip duration       | 2 – 240 min                         | Sub-2-min trips are likely app errors; 240 min (4hr) is implausible within NYC     |
| Trip distance       | 0.05 – 100 miles                    | Near-zero distances indicate failed GPS; 100 miles exceeds NYC boundaries          |
| Average speed       | < 80 mph; < 1 mph if trip > 10 min  | 80 mph is impossible on NYC surface roads; crawling trips indicate stalled records |
| Timestamp integrity | Dropoff must be after pickup        | Negative durations are data errors                                                 |
| Zone validity       | Location IDs must be in range 1–263 | TLC defines exactly 263 zones                                                      |
| Fare integrity      | No negative fare components         | TLC rules prohibit negative base fares, tolls, or taxes                            |

**Stage 2 — IQR outlier removal:** After fatal filtering, a monthly IQR filter is applied to fare amounts per zone. This removes statistical outliers while deliberately preserving high but plausible fares (e.g. JFK airport trips with surcharges). The filter is applied per month rather than globally to account for seasonal fare variation.


In [ ]:
# Filtering thresholds
MAX_TRIP_MINUTES = 240.0   # 4 hours — implausible for NYC
MIN_TRIP_MINUTES = 2.0     # below this is likely an app/recording error
MAX_TRIP_MILES = 100.0   # exceeds NYC boundaries
MIN_TRIP_MILES = 0.05    # near-zero distance = GPS failure
MAX_SPEED_MPH = 80.0    # impossible on surface roads
MIN_SPEED_MPH = 1.0     # crawling — only flagged if trip > 10 min
LOW_SPEED_MINLEN = 10.0    # minutes threshold for low-speed check


def build_violation_exprs(df):
    """Returns a dict of rule_name -> Boolean Column (True = violation to remove)."""
    exprs = {}

    # Duration rules
    if "trip_minutes" in df.columns:
        exprs["trip_minutes_neg"] = F.col("trip_minutes") < 0
        exprs["trip_minutes_too_short"] = (
            F.col("trip_minutes") < MIN_TRIP_MINUTES) & F.col("trip_minutes").isNotNull()
        exprs["trip_minutes_too_long"] = F.col(
            "trip_minutes") > MAX_TRIP_MINUTES

    # Distance rules
    if "trip_miles" in df.columns:
        exprs["trip_miles_neg"] = F.col("trip_miles") < 0
        exprs["trip_miles_too_short"] = (
            F.col("trip_miles") < MIN_TRIP_MILES) & F.col("trip_miles").isNotNull()
        exprs["trip_miles_too_long"] = F.col("trip_miles") > MAX_TRIP_MILES

    # Speed rules
    if "avg_speed_mph" in df.columns:
        exprs["speed_too_fast"] = F.col("avg_speed_mph") > MAX_SPEED_MPH
        exprs["speed_too_slow"] = (
            (F.col("avg_speed_mph") < MIN_SPEED_MPH) &
            (F.col("trip_minutes") > LOW_SPEED_MINLEN)
        )

    # Timestamp integrity
    if "pickup_ts_local" in df.columns and "dropoff_ts_local" in df.columns:
        exprs["dropoff_before_pickup"] = F.col(
            "dropoff_ts_local") <= F.col("pickup_ts_local")

    # Zone validity (TLC defines zones 1–263)
    for loc_col in ["pulocationid", "dolocationid"]:
        if loc_col in df.columns:
            exprs[f"{loc_col}_invalid"] = (
                F.col(loc_col).isNull() |
                (F.col(loc_col) < 1) |
                (F.col(loc_col) > 263)
            )

    # Fare integrity — TLC rules prohibit negative fare components
    for fare_col in ["base_passenger_fare", "tolls", "sales_tax", "driver_pay"]:
        if fare_col in df.columns:
            exprs[f"{fare_col}_neg"] = F.col(fare_col) < 0

    return exprs


def apply_fatal_filter(df):
    """Remove records violating any hard quality rule. Returns filtered df."""
    violation_exprs = build_violation_exprs(df)
    if not violation_exprs:
        return df
    combined_violation = reduce(lambda a, b: a | b, violation_exprs.values())
    return df.filter(~combined_violation)


def apply_iqr_filter(df, fare_col="base_passenger_fare", group_col="month"):
    """
    Monthly IQR filter on fare amounts.
    Removes statistical outliers while preserving high-but-plausible fares
    (e.g. airport trips with surcharges). Applied per month to account for
    seasonal fare variation.
    """
    quantiles = df.groupBy(group_col).agg(
        F.percentile_approx(fare_col, 0.25).alias("q1"),
        F.percentile_approx(fare_col, 0.75).alias("q3")
    ).withColumn("iqr", F.col("q3") - F.col("q1"))

    df = df.join(quantiles, on=group_col, how="left")
    df = df.filter(
        F.col(fare_col).between(
            F.col("q1") - 1.5 * F.col("iqr"),
            F.col("q3") + 1.5 * F.col("iqr")
        )
    ).drop("q1", "q3", "iqr")
    return df

### 2.6 Zone–Hour Aggregation & Profit Score

Individual trip records are aggregated to the **zone × hour** level — the unit of analysis for modelling. For each combination of pickup zone and hour, we compute:

- `n_trips` — total trip count (volume signal)
- `avg_duration`, `avg_miles`, `avg_speed_mph` — trip characteristic averages
- `profit_score` — the modelling target (see below)

**Profit score definition:**

$$\text{profit\_score}_{z,h} = \frac{\text{total driver pay}_{z,h}}{\text{total trip minutes}_{z,h}} \times n\_trips_{z,h}$$

Tips are deliberately excluded from driver pay — they are unpredictable and not guaranteed. This produces a conservative but consistent measure of earnings potential that reflects what a driver can reliably expect, not a best-case scenario.

**Lag features** are added to capture temporal demand persistence:

- `n_trips_t_1` — trips in the previous hour (immediate continuity)
- `n_trips_t_24` — trips at the same hour yesterday (daily rhythm)

Both features are computed strictly from past observations — no data leakage. At inference time, both values are already known.


In [ ]:
from pyspark.sql import Window


def aggregate_to_zone_hour(df):
    """Aggregate cleaned trip records to zone x hour level with profit score."""
    agg = df.groupBy("pulocationid", "pickup_hour_local", "year", "month").agg(
        F.count("*").alias("n_trips"),
        F.sum("trip_minutes").alias("sum_trip_minutes"),
        F.sum("driver_pay").alias("sum_driver_revenue"),
        F.avg("trip_minutes").alias("avg_duration"),
        F.avg("trip_miles").alias("avg_miles"),
        F.avg("avg_speed_mph").alias("speed_mph")
    )

    # Profit score: revenue efficiency × demand volume
    agg = agg.withColumn(
        "avg_revenue_per_min",
        F.when(F.col("sum_trip_minutes") > 0,
               F.col("sum_driver_revenue") / F.col("sum_trip_minutes")
               ).otherwise(F.lit(None))
    ).withColumn(
        "profit_score",
        F.col("avg_revenue_per_min") * F.col("n_trips")
    )
    return agg


def add_lag_features(df):
    """
    Add temporal lag features for n_trips.
    Both lags use only past observations — no data leakage.
    """
    zone_window = Window.partitionBy(
        "pulocationid").orderBy("pickup_hour_local")

    df = df.withColumn("n_trips_t_1",
                       F.lag("n_trips", 1).over(zone_window))   # previous hour
    df = df.withColumn("n_trips_t_24",
                       F.lag("n_trips", 24).over(zone_window))  # same hour yesterday
    return df

### 2.7 Pipeline Execution

The full pipeline is run across all three splits. Each split reads raw parquet files, applies normalisation, feature derivation, fatal filtering, IQR filtering, aggregation, and lag computation — then writes the result to partitioned parquet.

**Note:** This cell takes approximately 30 minutes to run in full.


In [ ]:
RAW_DIR = "../data/raw/fhv_hv"
PROCESSED_DIR = "../data/processed/fhv_hv/zone_hour_features_splits"

TRAIN_MONTHS = [(2024, m) for m in range(1, 5)]              # Jan–Apr 2024
VAL_MONTHS = [(2024, m) for m in range(5, 7)]              # May–Jun 2024
TEST_MONTHS = [(2025, m) for m in range(1, 7)]              # Jan–Jun 2025


def process_split(split_name, year_months, do_counts=True):
    """Run the full pipeline for a given split and write to parquet."""
    monthly_dfs = []

    for year, month in year_months:
        path = f"{RAW_DIR}/fhvhv_tripdata_{year}-{month:02d}.parquet"
        print(f"  → {year}-{month:02d} | reading {path}")

        df = spark.read.parquet(path)
        df = normalize_fhvhv(df, year=year)
        df = add_time_columns_local(df)
        df = add_trip_minutes(df)
        df = add_trip_avg_speed(df)
        df = fill_amount_nulls_with_zero(df)

        if do_counts:
            n_raw = df.count()
            print(f"    rows raw = {n_raw:,}")

        df = apply_fatal_filter(df)

        if do_counts:
            n_after_fatal = df.count()
            print(
                f"    rows after fatal filter = {n_after_fatal:,} (dropped {n_raw - n_after_fatal:,})")

        df = apply_iqr_filter(df)

        if do_counts:
            n_after_iqr = df.count()
            print(
                f"    rows after IQR = {n_after_iqr:,} (dropped {n_after_fatal - n_after_iqr:,} this step)")

        df = df.withColumn("year", F.lit(year)).withColumn(
            "month", F.lit(month))
        monthly_dfs.append(aggregate_to_zone_hour(df))

    combined = reduce(lambda a, b: a.unionByName(b), monthly_dfs)
    combined = add_lag_features(combined)

    out_path = f"{PROCESSED_DIR}/{split_name}"
    combined.write.mode("overwrite").partitionBy(
        "year", "month").parquet(out_path)
    print(f" wrote split '{split_name}' → {out_path}")


process_split("train", TRAIN_MONTHS, do_counts=True)
process_split("val",   VAL_MONTHS,   do_counts=True)
process_split("test",  TEST_MONTHS,  do_counts=True)

### 2.8 Filtering Results

After applying both filtering stages across both years:

| Year     | Raw rows        | After filtering | Dropped        | Drop rate  |
| -------- | --------------- | --------------- | -------------- | ---------- |
| 2024     | 120,864,668     | 108,725,626     | 12,139,042     | 10.04%     |
| 2025     | 120,995,191     | 108,044,497     | 12,950,694     | 10.70%     |
| **Both** | **241,859,859** | **216,770,123** | **25,089,736** | **10.37%** |

The ~10% drop rate is consistent across both years, which is a good sign — it suggests the filtering rules are stable and not overly sensitive to the policy change between years. A dramatically different drop rate in 2025 would have indicated the congestion pricing shift was affecting data quality, not just demand patterns.


### 2.9 Output Schema Verification

A quick sanity check on the output feature table for the first month of the test set.


In [ ]:
base = "../data/processed/fhv_hv/zone_hour_features_splits/test"
df_check = (
    spark.read
    .option("basePath", base)
    .parquet(f"{base}/year=2025/month=1")
)
df_check.show(3, truncate=False)

+------------+-------------------+-------+----------------+------------------+------------+---------+-------------------+--------------+-----------+------------+----+-----+
|pulocationid|pickup_hour_local  |n_trips|sum_trip_minutes|sum_driver_revenue|avg_duration|avg_miles|avg_revenue_per_min|profit_score  |n_trips_t_1|n_trips_t_24|year|month|
+------------+-------------------+-------+----------------+------------------+------------+---------+-------------------+--------------+-----------+------------+----+-----+
|12          |2025-01-01 00:00:00|6      |97.4            |122.38            |16.23       |5.55     |1.256              |7.539         |NULL       |NULL        |2025|1    |
|12          |2025-01-01 02:00:00|1      |11.15           |21.1              |11.15       |2.99     |1.892              |1.892         |6          |NULL        |2025|1    |
|12          |2025-01-01 04:00:00|2      |58.1            |71.61             |29.05       |11.64    |1.233              |2.465         

## 3. Weather Data Processing

Hourly weather conditions are included as features because adverse weather measurably affects both demand patterns and trip efficiency — rain increases ride requests while slowing average speeds, and extreme cold or heat shifts the timing of peak demand.

Weather data is sourced from the [NOAA Local Climatological Data portal](https://www.ncei.noaa.gov/oa/local-climatological-data/) for the Central Park station (USW00094728), which provides consistent hourly readings across NYC. The raw dataset contains a large number of columns — this section selects only the features with clear relevance to driving conditions and sufficient data completeness.


In [ ]:
from scripts.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = get_spark(app_name="02_process_weather")

In [ ]:
# Paths
ROOT = "../data/external/weather"
IN_2024 = f"{ROOT}/raw/LCD_USW00094728_2024.csv"
IN_2025 = f"{ROOT}/raw/LCD_USW00094728_2025.csv"

OUT_TRAIN = f"{ROOT}/processed/train_by_year_month"
OUT_VAL = f"{ROOT}/processed/val_by_year_month"
OUT_TEST = f"{ROOT}/processed/test_by_year_month"

### 3.1 Feature Selection

The raw NOAA dataset contains dozens of atmospheric variables, many with high missingness or low relevance to surface driving. We retain only four:

| Feature          | Raw column                  | Rationale                                           |
| ---------------- | --------------------------- | --------------------------------------------------- |
| `temp_c`         | `HourlyDryBulbTemperature`  | Affects comfort and demand timing                   |
| `dewpoint_c`     | `HourlyDewPointTemperature` | Proxy for humidity and perceived conditions         |
| `wind_speed_mps` | `HourlyWindSpeed`           | Affects trip efficiency and pedestrian behaviour    |
| `precip_mm`      | `HourlyPrecipitation`       | Strongest weather driver of rideshare demand spikes |

Wind direction was retained in intermediate processing but excluded from the final feature set — its effect on profitability is indirect and its circular encoding adds complexity without clear predictive value at the zone level.

NOAA raw values occasionally include trailing quality flags (e.g. `"72s"`) and trace precipitation markers (`"T"`). The `_num()` helper extracts the first numeric token, and trace precipitation is mapped to `0.0` — a trace amount is by definition negligible.


In [ ]:
def _num(colname):
    """Extract first numeric token as double — handles NOAA quality flag suffixes (e.g. '72s')."""
    return (
        F.when(F.col(colname).isNull(), None)
         .otherwise(
             F.regexp_extract(
                 F.col(colname), r"(-?\d+(?:\.\d+)?)", 1).cast("double")
        )
    )


def parse_ts(df):
    """Parse DATE column to timestamp, trying multiple common NOAA format patterns."""
    ts = F.coalesce(
        F.to_timestamp("DATE"),
        F.to_timestamp("DATE", "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp("DATE", "yyyy-MM-dd HH:mm"),
        F.to_timestamp("DATE", "MM/dd/yyyy HH:mm:ss"),
        F.to_timestamp("DATE", "MM/dd/yyyy HH:mm"),
    )
    return df.select(
        "DATE",
        "HourlyDryBulbTemperature",
        "HourlyDewPointTemperature",
        "HourlyWindSpeed",
        "HourlyWindDirection",
        "HourlyPrecipitation"
    ).withColumn("ts_local", ts)


def select_weather_features(df_ts):
    """Select and clean the four retained weather features."""
    return (
        df_ts
        .withColumn("hour_local", F.date_trunc("hour", F.col("ts_local")))
        .select(
            "hour_local",
            _num("HourlyDryBulbTemperature").alias("temp_c"),
            _num("HourlyDewPointTemperature").alias("dewpoint_c"),
            _num("HourlyWindSpeed").alias("wind_speed_mps"),
            # Trace precipitation ('T') mapped to 0.0 — negligible by definition
            F.regexp_extract(
                F.regexp_replace(F.col("HourlyPrecipitation"),
                                 r"(?i)T", "0.0"),
                r"(-?\d+(?:\.\d+)?)", 1
            ).cast("double").alias("precip_mm"),
        )
    )

### 3.2 Hourly Aggregation & Missingness Imputation

NOAA records multiple observations per hour from the same station. These are aggregated to a single row per hour using mean for continuous variables.

**Missingness handling:** Wind speed (8–10% missing) and precipitation (up to 6.7% missing) have non-trivial gaps. Rather than dropping affected rows — which would create gaps in the time series — missing values are forward-filled using the previous valid observation.

Forward-fill is appropriate here because weather conditions change gradually. A missing hour is far more likely to resemble the preceding hour than any global average, and dropping it would break the temporal continuity that the lag features in the trip data rely on.


In [ ]:
def aggregate_hourly(df_sel):
    """Aggregate to one row per hour using mean for continuous features."""
    return (
        df_sel
        .groupBy("hour_local")
        .agg(
            F.avg("temp_c").alias("temp_c"),
            F.avg("dewpoint_c").alias("dewpoint_c"),
            F.avg("wind_speed_mps").alias("wind_speed_mps"),
            F.sum("precip_mm").alias("precip_mm"),   # sum for precipitation
        )
        .orderBy("hour_local")
    )


def forward_fill(df, order_col, feature_cols):
    """
    Forward-fill missing values using the last valid observation.
    Applied per feature column over the ordered time series.
    Preferred over row-dropping to preserve temporal continuity.
    """
    window = Window.orderBy(order_col).rowsBetween(
        Window.unboundedPreceding, 0)
    for col in feature_cols:
        df = df.withColumn(
            col,
            F.last(F.col(col), ignorenulls=True).over(window)
        )
    return df

### 3.3 Pipeline Execution

The pipeline reads the raw 2024 and 2025 CSV files, applies parsing, feature selection, hourly aggregation, and forward-fill imputation, then writes the output to partitioned parquet split by train/val/test.


In [ ]:
FEATURE_COLS = ["temp_c", "dewpoint_c", "wind_speed_mps", "precip_mm"]

SPLITS = [
    ("train", "2024-01-01 00:00:00", "2024-05-01 00:00:00", OUT_TRAIN),
    ("val",   "2024-05-01 00:00:00", "2024-07-01 00:00:00", OUT_VAL),
    ("test",  "2025-01-01 00:00:00", "2025-07-01 00:00:00", OUT_TEST),
]


def process_weather_split(split_name, start_ts, end_ts, out_path):
    """Full weather pipeline for one split."""
    # Select source year based on split
    src = IN_2025 if start_ts.startswith("2025") else IN_2024

    df_raw = spark.read.option("header", True).csv(src)
    df_ts = parse_ts(df_raw)
    df_filt = df_ts.filter(
        (F.col("ts_local") >= F.to_timestamp(F.lit(start_ts))) &
        (F.col("ts_local") < F.to_timestamp(F.lit(end_ts)))
    )
    df_sel = select_weather_features(df_filt)
    df_agg = aggregate_hourly(df_sel)
    df_fill = forward_fill(df_agg, order_col="hour_local",
                           feature_cols=FEATURE_COLS)

    # Add year/month partition columns
    df_out = (
        df_fill
        .withColumn("year",  F.year("hour_local"))
        .withColumn("month", F.month("hour_local"))
    )

    df_out.write.mode("overwrite").partitionBy(
        "year", "month").parquet(out_path)
    print(f"  [{split_name}] written → {out_path}")


for split_name, start_ts, end_ts, out_path in SPLITS:
    process_weather_split(split_name, start_ts, end_ts, out_path)

### 3.4 Output Verification

Sample output from the test split to confirm schema and that imputation produced no remaining nulls.


In [ ]:
base = "../data/external/weather/processed/test_by_year_month"
df_check = (
    spark.read
    .option("basePath", base)
    .parquet(f"{base}/2025/01")
)
df_check.show(3, truncate=False)

# Confirm no nulls remain after imputation
null_counts = {c: df_check.filter(F.col(c).isNull()).count()
               for c in FEATURE_COLS}
print("Null counts after forward-fill:", null_counts)

+-------------------+------+-----------+--------------+---------+----+-----+
|hour_local         |temp_c|dewpoint_c |wind_speed_mps|precip_mm|year|month|
+-------------------+------+-----------+--------------+---------+----+-----+
|2025-01-01 00:00:00|2.2   |-3.9       |4.1           |0.0      |2025|1    |
|2025-01-01 01:00:00|1.7   |-4.4       |3.6           |0.0      |2025|1    |
|2025-01-01 02:00:00|1.1   |-5.0       |3.1           |0.0      |2025|1    |
+-------------------+------+-----------+--------------+---------+----+-----+
only showing top 3 rows


## 4. Calendar Feature Engineering

Travel demand in NYC follows strong and predictable temporal patterns — commuter peaks dominate weekdays, while weekends and public holidays shift demand toward evenings and leisure areas. Two binary flags capture this:

- **`is_weekend`** — Saturday or Sunday
- **`is_holiday`** — US federal public holiday (NY state, observed dates)

These are joined to the hourly feature table by date, giving the model a simple but effective signal for day-type effects without requiring it to learn them purely from hour-of-week patterns.

Holiday dates are generated using the [`holidays`](https://pypi.org/project/holidays/) Python library rather than a hardcoded list, making the pipeline easy to extend to additional years.


In [ ]:
from scripts.spark_session import get_spark
from pyspark.sql import functions as F, types as T, Row

spark = get_spark(app_name="03_process_calendar")

In [ ]:
# Output paths
BASE_OUT = "../data/external/calendar/processed"
OUT_TRAIN = f"{BASE_OUT}/train_by_year_month"
OUT_VAL = f"{BASE_OUT}/val_by_year_month"
OUT_TEST = f"{BASE_OUT}/test_by_year_month"

# Split windows — consistent with trip and weather splits
SPLITS = [
    ("train", "2024-01-01 00:00:00", "2024-05-01 00:00:00", OUT_TRAIN),
    ("val",   "2024-05-01 00:00:00", "2024-07-01 00:00:00", OUT_VAL),
    ("test",  "2025-01-01 00:00:00", "2025-07-01 00:00:00", OUT_TEST),
]

### 4.1 Holiday Calendar

The `holidays` library generates US federal holidays for NY state, respecting observed dates (e.g. when a holiday falls on a Sunday, the observed date is the following Monday). This is important for accurately capturing the demand shift — drivers and riders respond to the observed public holiday, not the nominal date.


In [ ]:
def build_holiday_calendar(start_date="2024-01-01", end_date="2025-07-01",
                           state="NY", observed=True):
    """
    Build a Spark DataFrame of public holiday dates within [start_date, end_date).
    Uses NY state federal holidays with observed date handling.
    """
    import datetime as dt
    import holidays as pyhol

    start = dt.date.fromisoformat(start_date)
    end = dt.date.fromisoformat(end_date)

    ny_holidays = pyhol.US(state=state, years=[2024, 2025], observed=observed)

    rows = [
        Row(date_local=d, is_holiday=1, holiday_name=name)
        for d, name in ny_holidays.items()
        if start <= d < end
    ]

    schema = T.StructType([
        T.StructField("date_local",   T.DateType(),    False),
        T.StructField("is_holiday",   T.IntegerType(), False),
        T.StructField("holiday_name", T.StringType(),  True),
    ])
    return spark.createDataFrame(rows, schema)

### 4.2 Hourly Calendar Grid

A complete hourly spine is generated for each split window — one row per hour with `is_weekend` and `is_holiday` flags attached. The holiday lookup is a simple left join on date; hours with no matching holiday date receive `is_holiday = 0`.

Encoding both flags as integers (1/0) rather than booleans keeps them compatible with the feature assembler used in modelling without additional casting.


In [ ]:
def build_hourly_calendar(start_ts, end_ts):
    """
    Generate a complete hourly time spine for the given window.
    Uses Spark's sequence() to avoid collecting to driver.
    """
    bounds = spark.createDataFrame(
        [(start_ts, end_ts)],
        schema=T.StructType([
            T.StructField("start_ts", T.StringType(), False),
            T.StructField("end_ts",   T.StringType(), False),
        ])
    )
    return (
        bounds
        .withColumn("start_ts_p", F.to_timestamp("start_ts"))
        .withColumn("end_ts_p",   F.to_timestamp("end_ts"))
        .withColumn("hours", F.expr(
            "sequence(start_ts_p, end_ts_p - interval 1 hour, interval 1 hour)"
        ))
        .select(F.explode("hours").alias("hour_local"))
    )


def add_calendar_flags(df_hourly, holiday_df, ts_col="hour_local"):
    """
    Attach is_weekend and is_holiday flags to each hour.
    - is_weekend: derived from day of week (1=Sun, 7=Sat in Spark)
    - is_holiday: left join on date; non-matching hours get 0
    """
    df = df_hourly.withColumn(
        "date_local", F.to_date(F.col(ts_col))
    ).withColumn(
        "is_weekend",
        F.when(F.dayofweek(F.col(ts_col)).isin(1, 7), 1).otherwise(0)
    )

    # Join holiday flags — unmatched hours default to is_holiday=0
    df = (
        df.join(holiday_df.select("date_local", "is_holiday"),
                on="date_local", how="left")
        .withColumn("is_holiday", F.coalesce(F.col("is_holiday"), F.lit(0)))
        .drop("date_local")
    )
    return df

### 4.3 Pipeline Execution


In [ ]:
# Build holiday lookup once — covers both 2024 and 2025
holiday_df = build_holiday_calendar(
    start_date="2024-01-01", end_date="2025-07-01")

for split_name, start_ts, end_ts, out_path in SPLITS:
    df_hourly = build_hourly_calendar(start_ts, end_ts)
    df_flags = add_calendar_flags(df_hourly, holiday_df)

    df_out = (
        df_flags
        .withColumn("year",  F.year("hour_local"))
        .withColumn("month", F.month("hour_local"))
    )

    df_out.write.mode("overwrite").partitionBy(
        "year", "month").parquet(out_path)
    print(f"  [{split_name}] written → {out_path}  ({df_out.count()} rows)")

### 4.4 Output Verification

Sample from the test split — confirming New Year's Day 2025 is correctly flagged.


In [ ]:
base = "../data/external/calendar/processed/test_by_year_month"
df_check = (
    spark.read
    .option("basePath", base)
    .parquet(f"{base}/2025/01")
)
df_check.show(5, truncate=False)

+-------------------+----------+----------+----+-----+
|hour_local         |is_weekend|is_holiday|year|month|
+-------------------+----------+----------+----+-----+
|2025-01-01 00:00:00|1         |1         |2025|1    |
|2025-01-01 01:00:00|1         |1         |2025|1    |
|2025-01-01 02:00:00|1         |1         |2025|1    |
|2025-01-02 00:00:00|0         |0         |2025|1    |
|2025-01-06 00:00:00|1         |0         |2025|1    |
+-------------------+----------+----------+----+-----+
only showing top 5 rows


## 5. Feature Join — Assembling the Final Dataset

This section brings together the three processed sources — trip features, weather, and calendar — into a single curated table ready for modelling.

The join key is `hour_local`: every zone–hour row in the trip table is matched to the corresponding hour's weather conditions and calendar flags. Two temporal features (`hour_of_day`, `day_of_week`) are also derived here and stored with the feature table.

**Leakage prevention** is a critical concern at this stage. The target variable (`profit_score`) and its direct components (`avg_revenue_per_min`, `n_trips`) are deliberately excluded from the feature table written to disk. They are stored separately and only joined at training time, ensuring the feature pipeline can never inadvertently expose target information.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from scripts.spark_session import get_spark

spark = get_spark(app_name="04_join_all_features")

In [ ]:
# Input paths — outputs from sections 2, 3, and 4
BASE_WEATHER_PROC = "../data/external/weather/processed"
BASE_CAL_PROC = "../data/external/calendar/processed"
BASE_TRIPS_PROC = "../data/processed/fhv_hv/zone_hour_features_splits"

# Curated output paths
CURATED_BASE = "../data/curated"
OUT_EXT_TRAIN = f"{CURATED_BASE}/external/train_by_year_month"
OUT_EXT_VAL = f"{CURATED_BASE}/external/val_by_year_month"
OUT_EXT_TEST = f"{CURATED_BASE}/external/test_by_year_month"
OUT_TRIPS_TRAIN = f"{CURATED_BASE}/fhvhv_with_features/train_by_year_month"
OUT_TRIPS_VAL = f"{CURATED_BASE}/fhvhv_with_features/val_by_year_month"
OUT_TRIPS_TEST = f"{CURATED_BASE}/fhvhv_with_features/test_by_year_month"

### 5.1 Reader Helpers

Each reader selects only the columns needed downstream, avoiding carrying unnecessary fields through the join pipeline. Weather and calendar are small tables (one row per hour) — these are broadcast-joined against the much larger trip table to avoid shuffle overhead.


In [ ]:
def read_weather(split):
    return (
        spark.read
             .option("recursiveFileLookup", "true")
             .parquet(f"{BASE_WEATHER_PROC}/{split}_by_year_month")
             .select("hour_local", "temp_c", "dewpoint_c", "wind_speed_mps", "precip_mm")
    )


def read_calendar(split):
    return (
        spark.read
             .option("recursiveFileLookup", "true")
             .parquet(f"{BASE_CAL_PROC}/{split}_by_year_month")
             .select("hour_local", "is_weekend", "is_holiday")
    )


def read_trips(split):
    """
    Read trip zone-hour features only — explicitly excludes target-adjacent columns
    (avg_revenue_per_min, n_trips) to prevent leakage into the feature table.
    """
    return (
        spark.read.parquet(f"{BASE_TRIPS_PROC}/{split}")
        .select(
            "pulocationid", "pickup_hour_local",
            "avg_duration", "avg_miles", "speed_mph",
            "n_trips_t_1", "n_trips_t_24",
        )
    )

### 5.2 Join Logic

Two joins are performed:

1. **Calendar + Weather → External features table** — joined on `hour_local`. Calendar is used as the spine since it has a complete hourly grid; weather rows with no matching hour (rare gaps) are filled via the earlier forward-fill step.
2. **Trips + External features → Final feature table** — broadcast join on `hour_local`. Broadcasting the external features table (small, ~4K rows per split) avoids a full shuffle of the trip table (~1M rows per split).

Temporal features (`hour_of_day`, `day_of_week`) are derived from the trip timestamp and stored alongside the other features.


In [ ]:
def build_external_features(split):
    """Join calendar (spine) with weather on hour_local."""
    cal = read_calendar(split)
    wx = read_weather(split)
    return cal.join(wx, on="hour_local", how="left").orderBy("hour_local")


def add_time_parts(df):
    """Derive hour_of_day and day_of_week from pickup timestamp."""
    return (
        df
        .withColumn("hour_of_day", F.hour("pickup_hour_local"))
        # 1=Sun, 7=Sat
        .withColumn("day_of_week", F.dayofweek("pickup_hour_local"))
    )


def join_trips_with_features(trips_df, feats_df):
    """Broadcast-join external features onto trip rows by hour."""
    return (
        trips_df
        .join(broadcast(feats_df),
              trips_df["pickup_hour_local"] == feats_df["hour_local"],
              how="left")
        .drop(feats_df["hour_local"])  # keep trips' timestamp as canonical
    )


def save_by_year_month(df, ts_col, base_out):
    """Write output partitioned by year/month, one file per month."""
    ym = (
        df.select(F.year(ts_col).alias("y"),
                  F.date_format(ts_col, "MM").alias("m"))
        .distinct().orderBy("y", "m").collect()
    )
    for r in ym:
        y, m = r["y"], r["m"]
        out = f"{base_out}/{y}/{m}"
        sub = df.filter(
            (F.year(ts_col) == y) & (F.date_format(ts_col, "MM") == m)
        ).coalesce(1)
        sub.write.mode("overwrite").parquet(out)
        print(f"  saved → {out}")


def print_shape(df, label):
    print(f"  {label}: ({df.count():,} rows, {len(df.columns)} cols)")


def match_rate(df, label, col="is_weekend"):
    total = df.count()
    matched = df.filter(F.col(col).isNotNull()).count()
    print(f"  {label} join match rate: {matched:,}/{total:,} ({matched/total:.1%})")

### 5.3 Stage 1 — External Features (Weather + Calendar)

First, weather and calendar are joined into a single external features table per split and written to the curated directory.


In [ ]:
for split, OUT_EXT in [
    ("train", OUT_EXT_TRAIN),
    ("val",   OUT_EXT_VAL),
    ("test",  OUT_EXT_TEST),
]:
    print(f"\n--- {split.upper()} ---")
    wx = read_weather(split)
    cal = read_calendar(split)
    print_shape(wx,  f"weather[{split}]")
    print_shape(cal, f"calendar[{split}]")

    ext = build_external_features(split)
    print_shape(ext, f"external_features[{split}]")
    save_by_year_month(ext, "hour_local", OUT_EXT)


--- TRAIN ---
  weather[train]: (2,902 rows, 5 cols)
  calendar[train]: (2,903 rows, 3 cols)
  external_features[train]: (2,903 rows, 7 cols)
  saved → ../data/curated/external/train_by_year_month/2024/01
  saved → ../data/curated/external/train_by_year_month/2024/02
  saved → ../data/curated/external/train_by_year_month/2024/03
  saved → ../data/curated/external/train_by_year_month/2024/04

--- VAL ---
  weather[val]: (1,463 rows, 5 cols)
  calendar[val]: (1,464 rows, 3 cols)
  external_features[val]: (1,464 rows, 7 cols)
  saved → ../data/curated/external/val_by_year_month/2024/05
  saved → ../data/curated/external/val_by_year_month/2024/06

--- TEST ---
  weather[test]: (4,306 rows, 5 cols)
  calendar[test]: (4,343 rows, 3 cols)
  external_features[test]: (4,343 rows, 7 cols)
  saved → ../data/curated/external/test_by_year_month/2025/01
  saved → ../data/curated/external/test_by_year_month/2025/02
  saved → ../data/curated/external/test_by_year_month/2025/03
  saved → ../data/curat

### 5.4 Stage 2 — Trip Features + External Features

The external features table is broadcast-joined onto the trip zone–hour table. A 100% match rate across all splits confirms there are no orphaned trip hours — every zone–hour row has a corresponding weather and calendar observation.


In [ ]:
for split, OUT_TRIPS in [
    ("train", OUT_TRIPS_TRAIN),
    ("val",   OUT_TRIPS_VAL),
    ("test",  OUT_TRIPS_TEST),
]:
    print(f"\n--- {split.upper()} ---")
    trips_raw = read_trips(split)
    trips_time = add_time_parts(trips_raw)
    print_shape(trips_raw, f"fhvhv[{split}]")

    ext = build_external_features(split)
    joined = join_trips_with_features(trips_time, ext)
    print_shape(joined, f"fhvhv+external[{split}]")
    match_rate(joined, split)

    # Drop target and its direct components — stored separately
    features_only = joined.drop("avg_revenue_per_min", "n_trips")
    save_by_year_month(features_only, "pickup_hour_local", OUT_TRIPS)


--- TRAIN ---
  fhvhv[train]: (727,047 rows, 7 cols)
  fhvhv+external[train]: (727,047 rows, 15 cols)
  train join match rate: 727,047/727,047 (100.0%)
  saved → ../data/curated/fhvhv_with_features/train_by_year_month/2024/01
  saved → ../data/curated/fhvhv_with_features/train_by_year_month/2024/02
  saved → ../data/curated/fhvhv_with_features/train_by_year_month/2024/03
  saved → ../data/curated/fhvhv_with_features/train_by_year_month/2024/04

--- VAL ---
  fhvhv[val]: (368,485 rows, 7 cols)
  fhvhv+external[val]: (368,485 rows, 15 cols)
  val join match rate: 368,485/368,485 (100.0%)
  saved → ../data/curated/fhvhv_with_features/val_by_year_month/2024/05
  saved → ../data/curated/fhvhv_with_features/val_by_year_month/2024/06

--- TEST ---
  fhvhv[test]: (1,089,557 rows, 7 cols)
  fhvhv+external[test]: (1,089,557 rows, 15 cols)
  test join match rate: 1,089,557/1,089,557 (100.0%)
  saved → ../data/curated/fhvhv_with_features/test_by_year_month/2025/01
  saved → ../data/curated/fhvhv_

### 5.5 Final Feature Schema

The curated feature table has 15 columns across ~1M zone–hour rows for the test period. This is the direct input to the modelling notebook.


In [ ]:
base = "../data/curated/fhvhv_with_features/test_by_year_month"
df_check = (
    spark.read
    .option("basePath", base)
    .parquet(f"{base}/2025/01")
)
df_check.show(3, truncate=False)

cols = df_check.columns
print(f"\nFinal feature columns ({len(cols)}): {', '.join(cols)}")

+------------+-------------------+------------+---------+-----------+-----------+------------+-----------+-----------+----------+----------+------+-----------+--------------+---------+
|pulocationid|pickup_hour_local  |avg_duration|avg_miles|speed_mph  |n_trips_t_1|n_trips_t_24|hour_of_day|day_of_week|is_weekend|is_holiday|temp_c|dewpoint_c |wind_speed_mps|precip_mm|
+------------+-------------------+------------+---------+-----------+-----------+------------+-----------+-----------+----------+----------+------+-----------+--------------+---------+
|12          |2025-01-01 00:00:00|16.23       |5.55     |20.50      |NULL       |NULL        |0          |4          |0         |1         |7.60  |6.30       |1.40          |0.0      |
|12          |2025-01-01 02:00:00|11.15       |2.99     |16.08      |6          |NULL        |2          |4          |0         |1         |7.33  |5.90       |0.70          |0.0      |
|12          |2025-01-01 04:00:00|29.05       |11.64    |24.03      |1     

## 6. Taxi Zone Geospatial Processing

This section builds the zone lookup table used for two purposes downstream:

1. **Choropleth maps** in the EDA notebook — each zone's geometry is needed to render profitability heatmaps over NYC
2. **Diversity metric** in the recommendation system — zone centroids are used to compute pairwise Haversine distances between recommended zones, penalising recommendations that cluster in the same neighbourhood

The raw TLC shapefiles use the NY State Plane coordinate system (EPSG:2263, units in feet). All geometry operations are performed in this projected CRS for accuracy, then reprojected to WGS84 (EPSG:4326, lat/lng) for Folium map rendering and distance calculations.


In [ ]:
import os
import pandas as pd
import geopandas as gpd
from scripts.spark_session import get_spark

spark = get_spark(app_name="05_process_taxi_zone")

# Paths
RAW_SHP = "../data/raw/taxi_zones/taxi_zones.shp"
RAW_LOOKUP = "../data/raw/taxi_zones/taxi_zone_lookup.csv"
CURATED_DIR = "../data/curated/geo"
OUT_GEOJSON = f"{CURATED_DIR}/nyc_taxi_zones_enriched.geojson"
OUT_CSV = f"{CURATED_DIR}/nyc_taxi_zones_lookup.csv"

### 6.1 Load Shapefiles & Zone Lookup

The TLC provides two complementary files:

- `taxi_zones.shp` — polygon geometries for all 263 zones
- `taxi_zone_lookup.csv` — zone names and borough assignments

The raw shapefile uses NY State Plane coordinates (large integer values in feet), which need reprojection before any lat/lng-based operations.


In [ ]:
sf = gpd.read_file(RAW_SHP)
zones = pd.read_csv(RAW_LOOKUP)

sf.head()

,OBJECTID,zone,LocationID,borough,geometry
0,1,Newark Airport,1,EWR,"POLYGON ((933100.918 192536.086, ..."
1,2,Jamaica Bay,2,Queens,"MULTIPOLYGON (((1033269.244 172126.008, ..."
2,3,Allerton/Pelham Gardens,3,Bronx,"POLYGON ((1026308.77 256767.698, ..."
3,4,Alphabet City,4,Manhattan,"POLYGON ((992073.467 203714.076, ..."
4,5,Arden Heights,5,Staten Island,"POLYGON ((935843.31 144283.336, ..."


### 6.2 CRS Reprojection & Centroid Computation

Two types of centre points are computed for each zone:

- **`centroid`** — geometric centroid of the polygon. For convex zones this is reliable, but for irregular shapes (e.g. Jamaica Bay) it can fall outside the polygon boundary.
- **`representative_point`** — guaranteed to lie inside the polygon. Used as the label anchor for map tooltips where visual placement matters.

Both are computed in EPSG:2263 (projected, metres-equivalent) for geometric accuracy, then converted to WGS84 lat/lng for downstream use.

Zone area is also computed in km² using the NY State Plane units (ft²), converted via the standard factor of 0.0929 ft²/m².


In [ ]:
# Merge shapefile with zone lookup on LocationID
gdf = sf.merge(
    zones[["LocationID", "Zone", "Borough"]].rename(
        columns={"LocationID": "pulocationid",
                 "Zone": "zone", "Borough": "borough"}
    ),
    left_on="LocationID", right_on="pulocationid", how="left"
)
gdf["pulocationid"] = gdf["pulocationid"].astype(int)

# Reproject to NY State Plane (EPSG:2263) for accurate planar geometry ops
gdf_xy = gdf.to_crs(2263)

# Compute both centroid types in projected CRS
# guaranteed inside polygon
gdf_xy["label_point"] = gdf_xy.geometry.representative_point()
gdf_xy["centroid_geom"] = gdf_xy.geometry.centroid

# Reproject points back to WGS84 (lat/lng) for map rendering and Haversine distances
labels_ll = gdf_xy.set_geometry("label_point").to_crs(4326)
centroid_ll = gdf_xy.set_geometry("centroid_geom").to_crs(4326)

gdf["centroid_lat"] = centroid_ll.geometry.y
gdf["centroid_lng"] = centroid_ll.geometry.x
gdf["rep_lat"] = labels_ll.geometry.y
gdf["rep_lng"] = labels_ll.geometry.x

# Area in km² (EPSG:2263 area is in ft²; 1 ft² = 0.09290304 m²)
gdf["area_km2"] = (gdf_xy.geometry.area * 0.09290304) / 1_000_000

# Select final output columns
cols_out = ["pulocationid", "zone", "borough",
            "centroid_lat", "centroid_lng", "rep_lat", "rep_lng",
            "area_km2", "geometry"]
gdf_out = gdf[[c for c in cols_out if c in gdf.columns]].copy()

# Save outputs
os.makedirs(CURATED_DIR, exist_ok=True)
gdf_out.to_file(OUT_GEOJSON, driver="GeoJSON")           # for Folium maps
gdf_out.drop(columns="geometry").to_csv(
    OUT_CSV, index=False)  # lightweight lookup for Spark

print("Saved:")
print(" -", OUT_GEOJSON)
print(" -", OUT_CSV)

# Sanity checks
print("ID dtype:", gdf_out["pulocationid"].dtype)
print(
    f"lat range : {gdf_out['centroid_lat'].min():.4f} → {gdf_out['centroid_lat'].max():.4f}")
print(
    f"lng range : {gdf_out['centroid_lng'].min():.4f} → {gdf_out['centroid_lng'].max():.4f}")
print(f"unique IDs: {gdf_out['pulocationid'].nunique()}  rows: {len(gdf_out)}")

Saved:
 - ../data/curated/geo/nyc_taxi_zones_enriched.geojson
 - ../data/curated/geo/nyc_taxi_zones_lookup.csv
ID dtype: int64
lat range : 40.5126 → 40.9116
lng range : -74.2557 → -73.7004
unique IDs: 263  rows: 263


### 6.3 Output Lookup Table

The enriched lookup table — 263 zones with centroid coordinates, representative points, area, and WGS84 geometry. This is the file joined to recommendation outputs for Haversine distance computation and map rendering.


In [ ]:
gdf_out.head()

,pulocationid,zone,borough,centroid_lat,centroid_lng,rep_lat,rep_lng,area_km2,geometry
0,1,Newark Airport,EWR,40.6918,-74.1740,40.6895,-74.1768,7.343,"POLYGON ((-74.18445 40.695, ..."
1,2,Jamaica Bay,Queens,40.6167,-73.8313,40.6257,-73.8261,13.370,"MULTIPOLYGON (((-73.82338 40.63899, ..."
2,3,Allerton/Pelham Gardens,Bronx,40.8645,-73.8474,40.8659,-73.8495,2.944,"POLYGON ((-73.84793 40.87134, ..."
3,4,Alphabet City,Manhattan,40.7238,-73.9770,40.7242,-73.9770,0.745,"POLYGON ((-73.97177 40.72582, ..."
4,5,Arden Heights,Staten Island,40.5527,-74.1885,40.5503,-74.1899,4.684,"POLYGON ((-74.17422 40.56257, ..."


---

## Pipeline Complete

All data sources are now processed and written to the curated directory. The outputs from this notebook feed directly into:

- **`02_eda.ipynb`** — choropleth maps using `nyc_taxi_zones_enriched.geojson`
- **`03_modelling.ipynb`** — feature table from `fhvhv_with_features/`
- **`04_recommendations.ipynb`** — diversity metric using `nyc_taxi_zones_lookup.csv` centroids

```
data/curated/
  geo/
    nyc_taxi_zones_enriched.geojson   ← 263 zones with WGS84 geometry
    nyc_taxi_zones_lookup.csv         ← lightweight centroid lookup
  external/
    {train,val,test}_by_year_month/   ← weather + calendar joined
  fhvhv_with_features/
    {train,val,test}_by_year_month/   ← full feature table (no target)
  profit_score_per_hour_zone/         ← target variable, stored separately
```
